#### 패키지 호출

In [ ]:
import pandas as pd

from com_fun import *

# 천 단위로 쉼표 추가하는 함수
def add_comma(x):
    return "{:,.0f}".format(x)

# 연령을 연령대로 환산
def age_categorize(age):
    age = (age // 10) * 10
    return age

# 시도_시군구 기준 데이터 불러오기
area_df = pd.read_excel('../P1_Data/01.External/시도_시군구_기준.xlsx')

#### 상품권 사용실적 전처리

In [ ]:
from openpyxl import load_workbook

# 엑셀 파일이 있는 폴더의 경로
excel_folder = "../P1_Data/01.External/결제내역/"

# 폴더 내의 모든 파일 목록 가져오기
files = os.listdir(excel_folder)

# 엑셀 파일을 CSV로 변환
for file in files:
    if file.endswith('.xlsx'):
        excel_file = os.path.join(excel_folder, file)
        csv_file = os.path.splitext(file)[0] + '.csv'

        # 엑셀 파일 읽기
        wb = load_workbook(excel_file)
        sheet = wb.active

        # 데이터프레임으로 변환하고 CSV로 저장
        df = pd.DataFrame(sheet.values)
        df.to_csv(os.path.join(excel_folder, csv_file), index=False, header=False)

        print(f'{file}을 {csv_file}로 변환 완료')

print('모든 엑셀 파일을 CSV로 변환 완료')

In [ ]:
# 변환한 CSV 파일들이 있는 폴더의 경로
csv_folder =  "../P1_Data/01.External/결제내역/"

# 폴더 내의 모든 파일 목록 가져오기
files = os.listdir(csv_folder)

# 빈 데이터프레임을 생성하여 CSV 파일들을 연결할 준비
combined_df = pd.DataFrame()

# CSV 파일을 읽고 데이터프레임에 추가
for file in files:
    if file.endswith('.csv'):
        csv_file = os.path.join(csv_folder, file)
        df = pd.read_csv(csv_file)

        # 데이터프레임을 연결
        combined_df = pd.concat([combined_df, df], ignore_index=True)

print('모든 CSV 파일을 하나의 데이터프레임으로 합침')

# 합쳐진 데이터프레임을 사용하거나 저장할 수 있습니다.
print(combined_df)

In [ ]:
# 필요없는 행 삭제
combined_df = combined_df.drop('열1', axis = 1)

# 결제일시 분리
combined_df[['결제일자', '결제시간']] = combined_df['결제일시'].str.split(' ', expand=True)

# 결제일자 시간속성 부여
combined_df['결제일자'] = pd.to_datetime(combined_df['결제일자'])

# 전처리 결과 저장
combined_df.to_pickle(f'../P1_Data/02.Pretreatment/pkl/ODS_Usage_history.pkl')

In [ ]:
combined_df = pd.read_pickle(f'../P1_Data/02.Pretreatment/pkl/ODS_Usage_history.pkl')
combined_df02 = combined_df.copy()
combined_df02 = combined_df02[['결제일자', '결제금액내역', '결제금액']]
combined_df02['결제연도'] = combined_df02['결제일자'].dt.year

# 연도별 결제금액 내역 순위 확인
combined_df02[['결제연도', '결제금액내역', '결제금액']].groupby(['결제연도', '결제금액내역']).agg({'sum'}).reset_index()

#### 상품권 가맹점 정보

In [ ]:
combined_df = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/ODS_Usage_history.pkl')
combined_df = combined_df.drop(['결제시간', '결제일시'], axis = 1)

# 가맹점 정보 불러오기
store_df = pd.read_csv('../P1_Data/01.External/모바일상품권가맹점정보.csv', encoding = 'utf-8')
#store_df = store_df[['가맹점ID', '가맹점상태', '가맹점명', '업종', '가맹점주소', '지자체명']]
store_df

In [ ]:
store_df = pd.read_csv('../P1_Data/01.External/모바일상품권가맹점정보.csv', encoding = 'utf-8')
store_df = store_df[['가맹점ID', '가맹점상태', '가맹점명', '업종', '가맹점주소', '지자체명']]
store_df = store_df.merge(area_df, left_on = '지자체명', right_on = '시군구명', how = 'left')

# 가맹점 데이터 결제 데이터 결합
merge_com_df = store_df.merge(combined_df, on = ['가맹점ID', '지자체명'])

# 데이터 프레임 저장
merge_com_df.to_pickle(f'../P1_Data/02.Pretreatment/pkl/merge_usage_store.pkl')

* 가맹점 정보 현황분석

In [ ]:
store_df = pd.read_csv('../P1_Data/01.External/모바일상품권가맹점정보.csv', encoding = 'utf-8')
# store_df = store_df[['지자체명', '가맹점명', '등록일자', '가맹점상태', '업종']]
store_df

In [ ]:
store_df['등록일자'] = pd.to_datetime(store_df['등록일자'])
store_df['가입연도'] = store_df['등록일자'].dt.year

# 지역 시도명 컬럼 추가
store_df02 = store_df.merge(area_df, left_on = '지자체명', right_on = '시군구명', how = 'left')
store_df02 = store_df02[['시도명', '지자체명', '가맹점명', '등록일자', '가맹점상태', '업종']]

# 연령대 분포 확인
a = store_df[store_df['가맹점상태'] == '가입완료'].업종.value_counts()#.reset_index()
a = pd.DataFrame(a)
a.applymap(add_comma).reset_index()
a

In [ ]:
# 가맹점 시군구 가입 비율
a = store_df.지자체명.value_counts()#.reset_index()
a = pd.DataFrame(a)
a.rename(columns = {'count' : '가맹점수'}, inplace = True)

# 비율구하기
a['비율'] = a['가맹점수'] / a['가맹점수'].sum()
a['비율'] = a['비율'].round(4)

# 단위
a.iloc[:,0:1] = a.iloc[:,0:1].applymap(add_comma)
a

In [ ]:
# 가맹점 시도 가입 비율
a = store_df02.시도명.value_counts()#.reset_index()
a = pd.DataFrame(a)
a.rename(columns = {'count' : '가맹점수'}, inplace = True)

# 비율구하기
a['비율'] = a['가맹점수'] / a['가맹점수'].sum()
a['비율'] = a['비율'].round(4)

# 단위
a.iloc[:,0:1] = a.iloc[:,0:1].applymap(add_comma)
a

In [ ]:
# 가맹점상태 확인
a = store_df.가맹점상태.value_counts()#.reset_index()
a = pd.DataFrame(a)
a.applymap(add_comma).reset_index()

In [ ]:
# 가입일자 확인
store_df['등록일자'] = pd.to_datetime(store_df['등록일자'])
store_df['가입연도'] = store_df['등록일자'].dt.year

# 연령대 분포 확인
a = store_df.가입연도.value_counts()#.reset_index()
a = pd.DataFrame(a)
a.applymap(add_comma).reset_index()

#### 상품권 판매량 데이터 전처리

In [ ]:
gift_sale = pd.read_excel('../P1_Data/01.External/상품권구매지급내역_가명처리.xlsx')

# 구매일자와 구매시간 분리
gift_sale[['구매_지급일', '구매_지급시간']] = gift_sale['구매_지급일자'].str.split(' ', expand=True)

# 시간 지정
gift_sale['구매_지급일'] = pd.to_datetime(gift_sale['구매_지급일'])

# 결측값 2행 제거
gift_sale = gift_sale.dropna(axis = 0).reset_index(drop = True)

# 시간분리
gift_sale['구매_연도'] = gift_sale['구매_지급일'].dt.year
gift_sale['구매_월'] = gift_sale['구매_지급일'].dt.month
gift_sale['구매_일'] = gift_sale['구매_지급일'].dt.day

gift_sale['구매금액'] = gift_sale['구매금액'].astype(int)

# 상품권 판매량 정보 저장
gift_sale.to_pickle(f'../P1_Data/02.Pretreatment/pkl/ODS_gift_history.pkl')

* 상품권 판매량 데이터 현황분석

In [ ]:
gift_sale = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/ODS_gift_history.pkl')
gift_sale = gift_sale[['지자체명', '수단구분', '구매금액', '할인금액', '총금액', '구매_연도', '구매_월']]

# 할인비율 구하기
gift_sale['할인비율'] = gift_sale['할인금액'] / gift_sale['총금액']
gift_sale_dis = gift_sale[gift_sale['할인금액'] != 0].copy()
gift_sale_dis

In [ ]:
# 총금액 기술통계
gift_sale['총금액구간'] = (gift_sale['총금액'] // 100000) * 100000
gift_sale['구매금액구간'] = (gift_sale['구매금액'] // 100000) * 100000

gift_sale[['총금액']].describe()

In [ ]:
gift_sale_buy = gift_sale[gift_sale['구매금액'] != 0].copy()
gift_sale_buy[['구매금액']].describe()

In [ ]:
# 금액 유형별 기술통계량
gift_sale_des = gift_sale[['구매금액', '할인금액', '총금액']].describe().applymap(add_comma)
#gift_sale_des.iloc[0:3,:] = gift_sale_des.iloc[0:3,:].applymap(add_comma)
gift_sale_des

In [ ]:
# 연도, 월별 구매금액 확인
gift_sale02 = gift_sale.copy()
gift_sale02[['구매_연도', '구매_월','구매금액']].groupby(['구매_연도', '구매_월']).sum().applymap(add_comma).reset_index()

In [ ]:
# 연도별 구매금액 확인
gift_sale02 = gift_sale.copy()
gift_sale02[['구매_연도','구매금액']].groupby(['구매_연도']).sum().applymap(add_comma).reset_index()

In [ ]:
gift_sale02 = gift_sale.copy()
gift_sale02[['구매_연도', '구매_월','총금액']].groupby(['구매_연도', '구매_월']).sum().applymap(add_comma).reset_index()

In [ ]:
gift_sale02 = gift_sale.copy()
gift_sale02[['구매_연도','총금액']].groupby(['구매_연도']).sum().applymap(add_comma).reset_index()

In [ ]:
# 수단구분 분포 확인
a = gift_sale.수단구분.value_counts()
a = pd.DataFrame(a)
a.rename(columns = {'count' : '수단구분'}, inplace = True)

# 비율구하기
a['비율'] = a['수단구분'] / a['수단구분'].sum()
a['비율'] = a['비율'].round(4)

# 단위
a.iloc[:,0:1] = a.iloc[:,0:1].applymap(add_comma)
a

In [ ]:
# 총 이용금액 현황분석
a = gift_sale[['수단구분', '총금액']].groupby(['수단구분']).sum().sort_values('총금액', ascending=False)
# 비율구하기
a['비율'] = a['총금액'] / a['총금액'].sum()
a['비율'] = a['비율'].round(4)

# 단위
a.iloc[:,0:1] = a.iloc[:,0:1].applymap(add_comma)
a

#### 사용자정보

In [ ]:
user_df = pd.read_excel('../P1_Data/01.External/사용자정보_가명처리.xlsx')
user_df = user_df.drop('핸드폰번호', axis = 1)

# 연령추가
user_df['연령'] = 2023 - user_df['생년월일'][user_df['생년월일'] >= 1900] + 1

# 연령대 추가
user_df['연령대'] = age_categorize(user_df['연령'])

# 시도추가
user_df02 = user_df.merge(area_df, left_on = '주거래지역', right_on = '시군구명', how = 'left')
user_df02 = user_df02[['회원번호', '성별','생년월일', '회원상태', '시도명', '주거래지역', '가입일자', '연령', '연령대']]

# 중복값 제거
user_df = user_df.drop_duplicates()

* 사용자정보 현황분석

In [ ]:
# 성별 ' '공백존재 그러나 회원상태가 탈퇴에 생년월일 1800년대로 아마 오류인듯
# 성별 분포 확인
a = user_df.성별.value_counts()#.reset_index()
a = pd.DataFrame(a)
a.applymap(add_comma).reset_index()

In [ ]:
# 회원상태 분포 확인
a = user_df.회원상태.value_counts()#.reset_index()
a = pd.DataFrame(a)
a.applymap(add_comma).reset_index()

In [ ]:
# 연령대 분포 확인
a = user_df.연령대.value_counts()#.reset_index()
a = pd.DataFrame(a)
a.applymap(add_comma).reset_index()

In [ ]:
# 연령대 분포 확인
a = user_df.주거래지역.value_counts()#.reset_index()
a = pd.DataFrame(a)
a.rename(columns = {'count' : '가입인원'}, inplace = True)

# 비율구하기
a['비율'] = a['가입인원'] / a['가입인원'].sum()
a['비율'] = a['비율'].round(4)

# 단위
a.iloc[:,0:1] = a.iloc[:,0:1].applymap(add_comma)
a

In [ ]:
# 연령대 분포 확인
a = user_df02.시도명.value_counts()#.reset_index()
a = pd.DataFrame(a)
a.rename(columns = {'count' : '가입인원'}, inplace = True)

# 비율구하기
a['비율'] = a['가입인원'] / a['가입인원'].sum()
a['비율'] = a['비율'].round(4)

# 단위
a.iloc[:,0:1] = a.iloc[:,0:1].applymap(add_comma)
a

In [ ]:
# 연령대 분포 확인
a = user_df02.value_counts(['성별', '시도명'])#.reset_index()
a = pd.DataFrame(a)
a.rename(columns = {'count' : '가입인원'}, inplace = True)

# 비율구하기
a['비율'] = a['가입인원'] / a['가입인원'].sum()
a['비율'] = a['비율'].round(4)

# 단위
a.iloc[:,0:1] = a.iloc[:,0:1].applymap(add_comma)
a

In [ ]:
user_df['가입일자'] = pd.to_datetime(user_df['가입일자'])
user_df['가입연도'] = user_df['가입일자'].dt.year

# 연령대 분포 확인
a = user_df.가입연도.value_counts()#.reset_index()
a = pd.DataFrame(a)
a.applymap(add_comma).reset_index()

In [ ]:
user_df02['가입일자'] = pd.to_datetime(user_df02['가입일자'])
user_df02['가입연도'] = user_df02['가입일자'].dt.year

user_df02 = user_df02.dropna()
# 연령대 분포 확인
user_df02[user_df02['성별']== 1].value_counts('가입연도')#.reset_index()

#### 분석 데이터셋 생성

In [ ]:
# 사용자 정보 불러오기
user_df = pd.read_excel('../P1_Data/01.External/사용자정보_가명처리.xlsx')
user_df = user_df.drop('핸드폰번호', axis = 1)

# 연령추가
user_df['연령'] = 2023 - user_df['생년월일'][user_df['생년월일'] >= 1900] + 1

# 연령대 추가
user_df['연령대'] = age_categorize(user_df['연령'])

# 중복값 제거
user_df = user_df.drop_duplicates()

In [ ]:
# 구매정보 불러오기
gift_sale = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/ODS_gift_history.pkl')
gift_sale = gift_sale.merge(area_df, left_on = '지자체명', right_on = '시군구명', how = 'left')

# 상품권 구매정보 + 사용자 정보 결합
merge_gift_user = gift_sale.merge(user_df, on = '회원번호')

merge_gift_user.to_pickle(f'../P1_Data/02.Pretreatment/pkl/merge_gift_user.pkl')
merge_gift_user

In [ ]:
# 상품권 이용정보 + 가맹점정보 결합한 데이터 불러오기
usage_store_df = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/merge_usage_store.pkl')

# 사용자정보 추가
merge_usage_store_user_df = usage_store_df.merge(user_df, on = ['회원번호'])

# 사용자정보 결합
merge_usage_store_user_df = usage_store_df.merge(user_df, on = ['회원번호'], how = 'left')
merge_usage_store_user_df.to_pickle(f'../P1_Data/02.Pretreatment/pkl/merge_usage_store_user.pkl')

#### 환불정보

In [ ]:
# 환불 정보 처리를 위해 구매내역 불러오기
gift_sale = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/ODS_gift_history.pkl')

# 유저정보 중복 제거 후 결합
user_df.drop_duplicates('회원번호').merge(re_gift_sale, on = '회원번호', how = 'inner')

In [ ]:
# 환불 내역 불러오기
re_gift_sale = pd.read_excel('../P1_Data/01.External/상품권환불내역_가명처리.xlsx')
re_gift_sale = re_gift_sale.dropna().reset_index(drop = True)
re_gift_sale

In [ ]:
re_gift_sale_cnt = re_gift_sale.회원번호.value_counts().reset_index()
re_gift_sale_cnt['구간'] = (re_gift_sale_cnt['count'] // 10) * 10
merge_re_gift_sale_cnt = re_gift_sale_cnt[['회원번호', '구간']].merge(re_gift_sale, on = '회원번호')
merge_re_gift_sale_cnt[merge_re_gift_sale_cnt['구간'] >= 30]

In [ ]:
# 평균환불 횟수 확인
merge_re_gift_sale_cnt_cnt = merge_re_gift_sale_cnt.회원번호.value_counts().reset_index()
merge_re_gift_sale_cnt_cnt['count'].mean()

In [ ]:
merge_re_gift_sale_cnt_cnt = merge_re_gift_sale_cnt.회원번호.value_counts().reset_index()
merge_re_gift_sale_cnt_cnt.rename(columns = {'count':'구매건수'}, inplace = True)
merge_re_gift_sale_cnt_cnt = merge_re_gift_sale_cnt_cnt.구매건수.value_counts().reset_index()
merge_re_gift_sale_cnt_cnt['비율'] = merge_re_gift_sale_cnt_cnt['count'] / merge_re_gift_sale_cnt_cnt['count'].sum()
merge_re_gift_sale_cnt_cnt['비율'] = merge_re_gift_sale_cnt_cnt['비율'].round(4)
merge_re_gift_sale_cnt_cnt

In [ ]:
re_gift_sale_cnt['비율'] = re_gift_sale_cnt['count'] / re_gift_sale_cnt['count'].sum()
re_gift_sale_cnt

In [ ]:
re_gift_sale02 = re_gift_sale[['회원번호', '지자체명', '환불일자', '환불금액', '환불할인금액', '환불총금액']]
re_gift_sale02 = re_gift_sale02.dropna()
re_gift_sale02[['환불일자_년월', '환불일자_시간']] = re_gift_sale02.환불일자.str.split(' ', expand = True)
re_gift_sale02['환불일자_년월'] = pd.to_datetime(re_gift_sale02['환불일자_년월'], format = '%Y-%m-%d')
re_gift_sale02['환불일자_연도'] = re_gift_sale02['환불일자_년월'].dt.year
re_gift_sale02['환불일자_월'] = re_gift_sale02['환불일자_년월'].dt.month
re_gift_sale02 = re_gift_sale02[['회원번호', '지자체명', '환불일자_년월', '환불일자_연도', '환불일자_월', '환불금액', '환불할인금액', '환불총금액']]
re_gift_sale02

In [ ]:
mer_re_gift_sale02 = re_gift_sale02.merge(re_mapping_01, on = '회원번호', how = 'inner')
mer_re_gift_sale02

In [ ]:
mer_gift_sale = gift_sale.merge(re_mapping_01, on = '회원번호', how = 'inner')
mer_gift_sale

In [ ]:
res_mer_re_01 = mer_gift_sale[['회원번호', '구매_지급일', '구매금액', '할인금액', '총금액']].merge(mer_re_gift_sale02[['회원번호', '환불일자_년월', '환불금액', '환불할인금액', '환불총금액']], on = '회원번호')
res_mer_re_01

In [ ]:
res_mer_re_01[~((res_mer_re_01['구매_지급일'] > res_mer_re_01['환불일자_년월']) | (res_mer_re_01['총금액'] <= res_mer_re_01['환불총금액']))]

#### 상품권 결제내역

In [ ]:
usage = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/ODS_Usage_history.pkl')
usage = usage.drop(['결제시간', '결제일시'], axis = 1)
usage = usage.dropna(axis = 0).reset_index(drop = True)

In [ ]:
# 수단구분 분포 확인
a = usage.사용수단.value_counts()
a = pd.DataFrame(a)
a.rename(columns = {'count' : '사용수단'}, inplace = True)

# 비율구하기
a['비율'] = a['사용수단'] / a['사용수단'].sum()
a['비율'] = a['비율'].round(4)

# 단위
a.iloc[:,0:1] = a.iloc[:,0:1].applymap(add_comma)
a

In [ ]:
usage_out = usage[usage['결제금액'] >= 2000000]
usage_out.결제금액내역.value_counts().reset_index()

In [ ]:
usage_out.사용수단.value_counts().reset_index()

In [ ]:
usage_out.결제_연도.value_counts().reset_index()

In [ ]:
# 시간분리
usage['결제_연도'] = usage['결제일자'].dt.year
usage['결제_월'] = usage['결제일자'].dt.month

In [ ]:
usage.결제금액.describe().transpose()

In [ ]:
usage.groupby(['결제_연도']).결제금액.describe().transpose()

In [ ]:
usage['결제금액구간'] = (usage['결제금액'] // 10000) * 10000

In [ ]:
usage_cnt = usage.결제금액구간.value_counts().reset_index()
usage_cnt.rename(columns = {'count' : '개수'}, inplace = True)
usage_cnt[usage_cnt['결제금액구간'] >= 200000].개수.sum()

In [ ]:
usage_cnt = usage[(usage['결제_연도'] == 2022)].결제금액구간.value_counts().reset_index()
usage_cnt.rename(columns = {'count' : '개수'}, inplace = True)
usage_cnt[usage_cnt['결제금액구간'] >= 200000].개수.sum()

In [ ]:
usage.groupby(['결제_연도', '결제금액구간']).결제금액.count().reset_index()#.transpose()

In [ ]:
usage.groupby(['결제_연도']).결제금액.describe().transpose()

In [ ]:
usage[['결제_연도', '결제_월', '결제금액']].groupby(['결제_연도', '결제_월']).agg(['mean', 'count']).reset_index()

In [ ]:
usage[['결제_연도', '결제금액']].groupby(['결제_연도']).agg(['mean', 'count']).reset_index()

In [ ]:
usage[['결제_연도', '결제_월', '결제금액']].groupby(['결제_연도', '결제_월']).count().reset_index()